# Simulate responses

{meth}`~liesel.model.Model.sample` draws from the distributions attached to a
model, following their dependencies. With posterior parameter draws supplied,
it can generate posterior predictive responses. It does not run posterior MCMC.

## Draw from the priors

Use `model` from {doc}`tutorials/notebooks/12-model-predictions`, whose mean
and log-scale coefficients have proper priors:

In [1]:
import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.goose as gs
import liesel.model as lsl

In [2]:
# Python cell fragment: imports are visible in each guide.
x_data = jnp.linspace(-1.0, 1.0, 120)
true_mean = 1.0 + 2.0 * x_data
true_scale = jnp.exp(-0.4 + 0.7 * x_data)
y_data = true_mean + true_scale * jax.random.normal(jax.random.key(8), (120,))

beta = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=2.5),
    name="beta",
)
gamma = lsl.Var.new_param(
    jnp.zeros(2),
    dist=lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="gamma",
)
x = lsl.Var.new_obs(x_data, name="x")
mu = lsl.Var.new_calc(
    lambda x, b: b[0] + b[1] * x,
    x,
    beta,
    name="mu",
)
log_sigma = lsl.Var.new_calc(
    lambda x, g: g[0] + g[1] * x,
    x,
    gamma,
    name="log_sigma",
)
sigma = lsl.Var.new_calc(jnp.exp, log_sigma, name="sigma")
y = lsl.Var.new_obs(
    y_data,
    dist=lsl.Dist(tfd.Normal, loc=mu, scale=sigma),
    name="y",
)
model = lsl.Model(y)

for parameter in model.parameters.values():
    parameter.inference = gs.MCMCSpec(
        gs.NUTSKernel,
        kernel_group="regression",
        jitter_dist=tfd.Normal(loc=0.0, scale=0.1),
    )
results = gs.LieselMCMC(model).run_for_epochs(
    seed=31,
    num_chains=4,
    adaptation=500,
    posterior=500,
    show_progress=False,
)
samples = results.get_posterior_samples()


liesel.goose.engine - INFO - Initializing kernels...


liesel.goose.engine - INFO - Done


liesel.goose.engine - INFO - Finished warmup


In [3]:
key = jax.random.key(27)
key, prior_key = jax.random.split(key)
prior_draws = model.sample(sample_shape=(6,), seed=prior_key)

In [4]:
prior_draws["y"].shape

(6, 120)

This returns six response datasets, with shape `(6, 120)`. Each uses newly
drawn coefficients and the existing covariates. These simulations reveal the
range of data allowed by your priors; very large or implausible values suggest
revisiting their scales. Use a fresh split key for another draw.

## Use posterior draws

For the posterior `samples` from the same tutorial:

In [5]:
x_grid = jnp.linspace(-1.0, 1.0, 60)

key, response_key = jax.random.split(key)
replicated = model.sample(
    sample_shape=(),
    seed=response_key,
    posterior_samples=samples,
    newdata={"x": x_grid, "y": jnp.zeros_like(x_grid)},
)

In [6]:
replicated["y"].shape

(4, 500, 60)

`sample_shape=()` requests one dataset per posterior draw. The result has shape
`(4, 500, 60)`. Posterior sample arrays must have two consistent leading axes
for chain and draw, followed by the parameter's own shape. The response
placeholder gives the new observation shape; random responses replace its values.

Supply all parameters you mean to condition on. A parameter with an attached
distribution but absent from `posterior_samples` may be sampled from that
distribution instead. Keys cannot occur in both `posterior_samples` and
`newdata`. See {doc}`model-prediction` for intervals on calculated quantities;
intervals computed from these replicated responses include response variation.

## Hold parameters fixed

To simulate responses conditional on the current coefficient values:

In [7]:
key, conditional_key = jax.random.split(key)
conditional = model.sample(
    sample_shape=(6,),
    seed=conditional_key,
    fixed=["beta", "gamma"],
)

In [8]:
conditional["y"].shape

(6, 120)

This keeps `beta` and `gamma` fixed and returns six datasets. Fixed entries
must not also appear in `posterior_samples`. Strong variables without distributions,
including fixed covariates, retain their supplied values. An unbounded parameter
without a distribution has an improper constant prior: retaining its current
value is conditional simulation, not drawing from that prior.

Calculated inputs are recomputed as needed. The returned dictionary contains
sampled quantities; use `predict` for derived quantities you want to inspect.
The `chunk_size` option limits parallel intermediate calculations, but all
returned samples still need to fit in memory.